# **Feature Selection with Boruta and Mother+**
This notebook demonstrates how to use Boruta feature selection with the Mother+ package on a synthetic dataset where only a few features are informative.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
from matplotlib.patches import Patch
from mother.ml.estimators import MotherCatboostImportance, MotherPermutationImportance, MotherBorutaPy
from scipy import stats
import mother.pipeline_utils as mother_takes_care
from sklearn.model_selection import GroupKFold
from mother.settings import MotherSettings

# Set random seed for reproducibility
np.random.seed(42)


# Define visualization functions
def plot_feature_correlation_with_target(correlations):
    """Plot feature correlations with the target variable."""
    plt.figure(figsize=(10, 8))
    plt.barh(y=correlations["feature"], width=correlations["correlation_with_target"])
    plt.xlabel("Correlation with Target")
    plt.title("Feature Correlation with Target Variable")
    plt.grid(axis="x")
    plt.tight_layout()
    plt.show()


def plot_feature_importances(importances, feature_names, support):
    """Plot feature importances with color coding for selected/rejected features."""
    # Sort features by importance
    sorted_idx = np.argsort(importances)
    feature_names_sorted = feature_names[sorted_idx]
    importances_sorted = importances[sorted_idx]
    support_sorted = support[sorted_idx]

    # Plot feature importances
    plt.figure(figsize=(12, 8))
    bars = plt.barh(range(len(sorted_idx)), importances_sorted, align="center")

    # Color by support status (green = confirmed, red = rejected)
    for i, is_supported in enumerate(support_sorted):
        bars[i].set_color("green" if is_supported else "red")

    plt.yticks(range(len(sorted_idx)), feature_names_sorted)
    plt.xlabel("Feature Importance")
    plt.title("Feature Importances (Green = Confirmed, Red = Rejected)")
    plt.grid(axis="x")
    plt.tight_layout()
    plt.show()


def plot_importance_comparison(comparison_df, catboost_selected_col, permutation_selected_col):
    """Plot comparison of CatBoost vs Permutation importance methods."""
    # Get top 10 features by average importance
    top_features = comparison_df.head(10)["Feature"].to_numpy()

    # Prepare data for plotting
    plot_data = comparison_df[comparison_df["Feature"].isin(top_features)].sort_values("Average Z-Score")
    features = plot_data["Feature"].to_numpy()
    x = np.arange(len(features))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 8))

    # Plot bars using z-scores
    catboost_bars = ax.barh(x - width / 2, plot_data["CatBoost Importance (Z)"], width, label="CatBoost")
    permutation_bars = ax.barh(x + width / 2, plot_data["Permutation Importance (Z)"], width, label="Permutation")

    # Color bars based on selection status
    for i, (cat_sel, perm_sel) in enumerate(zip(plot_data[catboost_selected_col], plot_data[permutation_selected_col])):
        catboost_bars[i].set_color("green" if cat_sel else "red")
        permutation_bars[i].set_color("blue" if perm_sel else "orange")

    # Add labels and legend
    ax.set_yticks(x)
    ax.set_yticklabels(features)
    ax.set_xlabel("Feature Importance (Z-Score)")
    ax.set_title("CatBoost vs Permutation Importance for Top Features (Z-Scores)")

    # Add a vertical line at Z=0 (mean importance)
    ax.axvline(x=0, color="black", linestyle="--", alpha=0.7, label="Mean Importance")

    # Create custom legend
    legend_elements = [
        Patch(facecolor="green", label="CatBoost - Selected"),
        Patch(facecolor="red", label="CatBoost - Rejected"),
        Patch(facecolor="blue", label="Permutation - Selected"),
        Patch(facecolor="orange", label="Permutation - Rejected"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")

    plt.tight_layout()
    plt.show()


def analyze_importance_methods(
    cat_importances, perm_importances, comparison_df, catboost_selected_col, permutation_selected_col
):
    """Analyze and compare feature importance methods."""
    # Calculate correlation between importance methods (using raw values)
    corr = np.corrcoef(cat_importances, perm_importances)[0, 1]
    print(f"\nCorrelation between CatBoost and Permutation importances: {corr:.4f}")

    # Calculate agreement rate on feature selection
    agreement = (comparison_df[catboost_selected_col] == comparison_df[permutation_selected_col]).mean() * 100
    print(f"Agreement on feature selection: {agreement:.1f}%")

    # Compare rankings
    cat_ranks = pd.Series(cat_importances).rank(ascending=False)
    perm_ranks = pd.Series(perm_importances).rank(ascending=False)
    rank_corr = cat_ranks.corr(perm_ranks, method="spearman")
    print(f"Spearman rank correlation: {rank_corr:.4f}")

    # More detailed analysis on differences (using z-scores)
    print("\nFeatures with biggest z-score difference:")
    comparison_df["Z-Score Difference"] = abs(
        comparison_df["CatBoost Importance (Z)"] - comparison_df["Permutation Importance (Z)"]
    )
    biggest_diff = comparison_df.sort_values("Z-Score Difference", ascending=False).head(5)
    print(
        biggest_diff[
            [
                "Feature",
                "CatBoost Importance (Z)",
                "Permutation Importance (Z)",
                catboost_selected_col,
                permutation_selected_col,
                "Z-Score Difference",
            ]
        ]
    )

    # Stability analysis - would require multiple runs but here's a placeholder comment:
    print("\nNote: For a complete stability analysis, multiple runs with different random seeds would be needed.")
    print("Permutation importance is generally more stable across runs but more computationally expensive.")
    print("CatBoost importance is faster but may be more sensitive to the specific model instance.")

1. Creating a Synthetic Dataset
We'll create a dataset with 20 features but only 5 of them will be informative:

In [ ]:
# Generate a synthetic classification dataset
X, y = make_classification(
    n_samples=1000,  # Number of samples
    n_features=20,  # Total number of features
    n_informative=5,  # Only 5 features are informative
    n_redundant=5,  # 5 features are redundant (derived from informative)
    n_repeated=0,  # No repeated features
    n_classes=2,  # Binary classification
    flip_y=0.1,  # 10% noise in labels
    random_state=42,
)

# Convert to dataframe for better visualization
column_names = [f"feature_{i + 1}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=column_names)
y_series = pd.Series(y, name="target")

# Display the first few rows of the dataset
print(f"Dataset shape: {X_df.shape}")
X_df.head()

2. Exploring the Dataset
Let's look at some basic statistics of our features:

In [ ]:
# Summary statistics
X_df.describe()

# Correlation with target
correlations = pd.DataFrame(
    {"feature": column_names, "correlation_with_target": [np.corrcoef(X_df[col], y)[0, 1] for col in column_names]}
)
correlations = correlations.sort_values("correlation_with_target", key=abs, ascending=False)

# Visualize correlations using the function
plot_feature_correlation_with_target(correlations)

3. Feature Selection with Boruta and Mother
Now, let's use Boruta with Mother's feature importance estimators:

In [ ]:
# Create a CatBoost base estimator
cat_estimator = CatBoostClassifier(iterations=100, learning_rate=0.1, max_depth=5, random_seed=42, verbose=False)

# Create a MotherCatboostImportance estimator
cat_imp = MotherCatboostImportance(
    estimator=cat_estimator,
    percentiles=False,  # Boruta works better with raw importance values
    n_estimators=100,
)

# Initialize Boruta with the feature importance estimator
boruta_selector = MotherBorutaPy(
    estimator=cat_imp,
    n_estimators="auto",  # Let Boruta determine the number of iterations
    verbose=2,  # Print progress logs
    random_state=42,
)

# Fit Boruta to the data
boruta_selector.fit(X_df, y)

# Print the results
print("\nFeature selection results:")
for feature, support, rank in zip(X_df.columns, boruta_selector.support_, boruta_selector.ranking_):
    status = "Confirmed" if support else "Rejected"
    print(f"Feature: {feature:<15} Status: {status:<10} Rank: {rank}")

4. Visualizing the Results
Let's visualize the feature importances and selected features:

In [ ]:
# Get feature importances
# Train an independent MotherCatboostImportance estimator
independent_imp = MotherCatboostImportance(
    estimator=cat_estimator,
    percentiles=False,
    n_estimators=100,
)
independent_imp.fit(X_df, y)
importances = independent_imp.feature_importances_
feature_names = np.array(column_names)

# Visualize feature importances using the function
plot_feature_importances(importances, feature_names, boruta_selector.support_)

5. Evaluating with Selected Features
Let's evaluate model performance using only the selected features:

In [ ]:
# Get selected features
selected_features = X_df.columns[boruta_selector.support_]
print(f"Selected features: {', '.join(selected_features)}")

# Transform the dataset to keep only selected features
X_selected = X_df[selected_features]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.3, random_state=42)

# Train a model with only selected features
model = CatBoostClassifier(iterations=200, learning_rate=0.1, max_depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy with {len(selected_features)} selected features: {accuracy:.4f}")

# Compare with model using all features
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_df, y, test_size=0.3, random_state=42)
model_full = CatBoostClassifier(iterations=200, learning_rate=0.1, max_depth=5, random_seed=42, verbose=False)
model_full.fit(X_train_full, y_train_full)
y_pred_full = model_full.predict(X_test_full)
accuracy_full = accuracy_score(y_test_full, y_pred_full)
print(f"Model accuracy with all {X_df.shape[1]} features: {accuracy_full:.4f}")

6. Using Permutation Importance
Let's also try with MotherPermutationImportance for comparison:

In [ ]:
# Create a MotherPermutationImportance estimator
perm_imp = MotherPermutationImportance(estimator=cat_estimator, percentiles=False, n_estimators=5, random_state=42)

# Initialize Boruta with the permutation importance estimator
boruta_selector_perm = MotherBorutaPy(estimator=perm_imp, n_estimators="auto", verbose=2, random_state=42)

# Fit Boruta to the data
boruta_selector_perm.fit(X_df, y)

# Print the results
print("\nFeature selection results (Permutation Importance):")
for feature, support, rank in zip(X_df.columns, boruta_selector_perm.support_, boruta_selector_perm.ranking_):
    status = "Confirmed" if support else "Rejected"
    print(f"Feature: {feature:<15} Status: {status:<10} Rank: {rank}")

# Compare selected features between methods
cat_selected = set(X_df.columns[boruta_selector.support_])
perm_selected = set(X_df.columns[boruta_selector_perm.support_])

print("\nComparison of selected features:")
print(f"Features selected by both methods: {cat_selected.intersection(perm_selected)}")
print(f"Features selected only by CatBoost: {cat_selected - perm_selected}")
print(f"Features selected only by Permutation: {perm_selected - cat_selected}")

7. Conclusion
In this notebook, we demonstrated:

How to create a synthetic dataset with informative and noise features
How to use Mother's feature importance estimators with Boruta for feature selection
How to visualize and interpret the feature selection results
How to evaluate model performance using only selected features
How different feature importance methods can lead to slightly different feature selections
Boruta with Mother+ provides a robust way to identify truly informative features, which can help improve model performance and interpretability by removing noise features.

In [ ]:
# 7. Comparing CatBoost vs Permutation Importance

# Define constants to avoid string duplication
CATBOOST_SELECTED = "CatBoost Selected"
PERMUTATION_SELECTED = "Permutation Selected"

# Train independent CatBoost and Permutation importance estimators
cat_imp_indep = MotherCatboostImportance(
    estimator=cat_estimator, max_depth=5, n_estimators=89, percentiles=False, random_state=42
)
cat_imp_indep.fit(X, y)
cat_importances = cat_imp_indep.feature_importances_

perm_imp_indep = MotherPermutationImportance(
    estimator=cat_estimator, max_depth=5, n_estimators=89, percentiles=False, random_state=42
)
perm_imp_indep.fit(X, y)
perm_importances = perm_imp_indep.feature_importances_


# Calculate z-scores for both sets of importances to make them comparable
cat_importances_z = stats.zscore(cat_importances)
perm_importances_z = stats.zscore(perm_importances)

# Create a DataFrame for comparison
comparison_df = pd.DataFrame(
    {
        "Feature": column_names,
        "CatBoost Importance (Raw)": cat_importances,
        "Permutation Importance (Raw)": perm_importances,
        "CatBoost Importance (Z)": cat_importances_z,
        "Permutation Importance (Z)": perm_importances_z,
    }
)

# Define selection as top features by Boruta (True if feature in selected_features)
comparison_df[CATBOOST_SELECTED] = comparison_df["Feature"].isin(selected_features)
comparison_df[PERMUTATION_SELECTED] = comparison_df["Feature"].isin(selected_features)

# Add a column showing if selection differs between methods
comparison_df["Selection Match"] = comparison_df[CATBOOST_SELECTED] == comparison_df[PERMUTATION_SELECTED]

# Sort by average z-score importance
comparison_df["Average Z-Score"] = (
    comparison_df["CatBoost Importance (Z)"] + comparison_df["Permutation Importance (Z)"]
) / 2
comparison_df = comparison_df.sort_values("Average Z-Score", ascending=False)

# Display the comparison table
print("Feature Importance Comparison (Z-Scores):")
comparison_df[
    [
        "Feature",
        "CatBoost Importance (Z)",
        "Permutation Importance (Z)",
        CATBOOST_SELECTED,
        PERMUTATION_SELECTED,
        "Selection Match",
    ]
]

# Visualize CatBoost vs Permutation importance methods
plot_importance_comparison(comparison_df, CATBOOST_SELECTED, PERMUTATION_SELECTED)

# Analyze importance methods
analyze_importance_methods(cat_importances, perm_importances, comparison_df, CATBOOST_SELECTED, PERMUTATION_SELECTED)

8. How to use Boruta with the Mother takes Care pipeline

In [ ]:
# Using Mother's get_feature_selection_pipeline with Boruta


# Create cross-validation strategy
cv = GroupKFold(n_splits=5)

# Configure settings for feature selection with Boruta
settings: MotherSettings = MotherSettings.create()
settings.model.feature_selection_flags = [
    "DROP_CONSTANT",
    "DROP_CORRELATED",
    "DROP_DUPLICATES",
    "DROP_UNIMPORTANT",
]

# Prepare a classification dataset for X_chem and y_chem
# Use the synthetic dataset X_df and y_series, and add a 'smiles' column for compatibility
X = X_df.copy()
y = y_series.values

# Create a Mother takes care pipeline with Boruta
# Set use_boruta=True to enable Boruta feature selection
pipeline = mother_takes_care.get_feature_selection_pipeline(
    settings=settings, data=X, use_boruta=True, cv=cv
).set_output(transform="pandas")


# Fit the pipeline to your data
pipeline.fit_transform(X, y)